# MCP服务器
---
本节，我们将在 LangGraph 中接入 MCP Server。在接入 MCP Server 之前，必须得有 MCP Server。也就是说，我们得开发一个 MCP Server。

## 一、开发 MCP 服务

### 1）天气 MCP

以 `get_weather_mcp` 为例，我们要把这个 MCP 写成一个 Python 包供，本地使用.

为了让它被识别为 Python 包，我们要在项目下，新建一个 `__init__.py` 文件。然后把主逻辑写在 `server.py` 中，接着在 `__main__.py` 中使用 `from . import server` 引入它。最后用 streamable-http 的方式部署它：

```python
def http():
    """streamable-http entry point for the package."""
    asyncio.run(server.mcp.run(transport="http",
                               host=host,
                               port=port,
                               path="/mcp"))
```

写到这里就齐活了。这里使用 `__main__.py` 是有小巧思的，这样我们可以将这个包作为模块直接在命令行使用。什么意思呢？就是我们用 `python -m [包名]` 就等于直接运行了 `__main__.py` 这个特殊文件。由于我们先前在该特殊文件中启动了 `http()` 函数，这样就能快捷方便地把 MCP Server 启动起来了！对于我们的 `get_weather_mcp`，启动命令如下：

```bash
python -m get_weather_mcp
```

### 2）算数 MCP

这还需要赘述吗？开发流程照抄上面的步骤。

真的是超级模版化。`__init__.py` 和 `__main__.py` 几乎完全相同。

唯一需要改动的是 `__main__.py`。需要把端口 `port` 改成新号码，一般来说加 1 就行。这里我们把 8000 改成 8001，其他不变：

```python
# -*- coding: utf-8 -*-
import asyncio
import os

from . import server


host = os.getenv('HOST', '127.0.0.1')
port = int(os.getenv('PORT', 8001))


def stdio():
    """Stdio entry point for the package."""
    asyncio.run(server.mcp.run(transport="stdio"))


def http():
    """streamable-http entry point for the package."""
    asyncio.run(server.mcp.run(transport="http",
                               host=host,
                               port=port,
                               path="/mcp"))


if __name__ == "__main__":
    http()
```

### 二、使用 `supervisord` 管理 MCP 服务

[supervisord](https://github.com/Supervisor/supervisor) 是一个 进程管理工具。你告诉它有哪些 MCP 要跑，它会守护你的 MCP 。当 MCP 挂掉的时候，supervisord 能够自动拉起 MCP。

首先，我们打开项目的 `mcp_server` 路径，在这里创建一个配置文件 `mcp_supervisor.conf`，来给 `supervisord` 使用。我的配置如下：

```
[unix_http_server]
file=/tmp/supervisor.sock

[supervisord]
logfile=/tmp/supervisord.log
logfile_maxbytes=50MB
logfile_backups=10
loglevel=info
pidfile=/tmp/supervisord.pid
nodaemon=false
minfds=1024
minprocs=200

[rpcinterface:supervisor]
supervisor.rpcinterface_factory = supervisor.rpcinterface:make_main_rpcinterface

[supervisorctl]
serverurl=unix:///tmp/supervisor.sock

[program:math_mcp]
command=python -m mcp_server.math_mcp
directory=..
autostart=true
autorestart=true
startsecs=5
stopwaitsecs=10
stdout_logfile=/tmp/math_mcp.log
stderr_logfile=/tmp/math_mcp_err.log

[program:weather_mcp]
command=python -m mcp_server.get_weather_mcp
directory=..
autostart=true
autorestart=true
startsecs=5
stopwaitsecs=10
stdout_logfile=/tmp/weather_mcp.log
stderr_logfile=/tmp/weather_mcp_err.log

[group:mcp_servers]
programs=math_mcp,weather_mcp
```

至此，`math_mcp`、`weather_mcp` 的配置就完成了。

### 1）安装 `supervisord`

```bash
pip install supervisor
```

### 2）启动 `supervisord`

```bash
supervisord -c ./mcp_supervisor.conf
```

### 3）关闭 `supervisord`

```bash
pkill -f supervisord
```

### 4）检查端口状态

```bash
lsof -i :8000
lsof -i :8001
```

In [29]:
from importlib.metadata import version
print("langchain version: ", version("langchain"))
print("langgraph version: ", version("langgraph"))
print("supervisor version: ", version("supervisor"))
print("langchain-mcp-adapters version: ", version("langchain-mcp-adapters"))
print("langgraph-supervisor version: ", version("langgraph-supervisor"))

langchain version:  1.2.10
langgraph version:  1.0.9
supervisor version:  4.3.0
langchain-mcp-adapters version:  0.2.1
langgraph-supervisor version:  0.0.31


### 2）接入 MCP 服务

使用 `MultiServerMCPClient` 接入 MCP Server.

In [35]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient  
from langchain.agents import create_agent

# 加载模型配置
_ = load_dotenv()

# 加载模型
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model="qwen3-max",
    temperature=0.7,
)

async def mcp_agent():
    # 我们用两种方式启动 MCP Server：stdio 和 streamable_http
    client = MultiServerMCPClient(  
        {
            "math": {
                "command": "python",
                "args": [os.path.abspath("./mcp_server/math_mcp/server.py")],
                "transport": "stdio",
            },
            "weather": {
                "url": "http://localhost:8000/mcp",
                "transport": "streamable_http",
            }
        }
    )
    
    tools = await client.get_tools()
    agent = create_agent(
        llm,
        tools=tools,
    )

    return agent

async def use_mcp(messages):
    agent = await mcp_agent()
    response = await agent.ainvoke(messages)
    return response

在 Jupyter Notebook 中，使用 `response = await use_mcp(messages)` 命令调用函数。但是在 `.py` 文件中，这种调用方法会失败。

In [36]:
# 调用天气 MCP
messages = {"messages": [{"role": "user", "content": "长沙天气怎么样？"}]}
response = await use_mcp(messages)
response["messages"][-1].content

'长沙天气晴朗，万里碧空飘着朵朵白云！适合外出活动或享受阳光哦！'

In [34]:
# 调用算数 MCP，由于是 stdio，启动会慢一点
messages = {"messages": [{"role": "user", "content": "计算 (3 + 5) * 12"}]}
response = await use_mcp(messages)
response["messages"][-1].content

'计算结果是 96。'

# 注意

在 `.py` 文件中，应该使用 `asyncio`，改动部分如下：

```python
import asyncio

async def main():
    # 调用天气 MCP
    messages = {"messages": [{"role": "user", "content": "福州天气怎么样？"}]}
    response = await use_mcp(messages)
    print(response["messages"][-1].content)

if __name__ == "__main__":
    asyncio.run(main())

```